# Введение в MapReduce модель на Python


In [2]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [3]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [4]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [5]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [6]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [7]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [8]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [9]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [10]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [11]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [12]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [13]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных. 

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [14]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*
 
mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL 

In [15]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str
    
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)
    
def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)
 
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication 

In [16]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])
 
def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])
      
output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(1.0995091973273068)),
 (1, np.float64(1.0995091973273068)),
 (2, np.float64(1.0995091973273068)),
 (3, np.float64(1.0995091973273068)),
 (4, np.float64(1.0995091973273068))]

## Inverted index 

In [17]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)
      
def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)
 
def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('it', ['0', '1', '2']),
 ('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [18]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [19]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()
      
def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]
 
def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers
  
def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)
  
  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*
 
e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*
 
flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount 

In [20]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)
      
  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):  
    yield (word, 1)
 
def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)
  
# try to set COMBINER=REDUCER and look at the number of values sent over the network 
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None) 
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('is', 18), ('it', 18), ('what', 10)]),
 (1, [('a', 2), ('banana', 2)])]

## TeraSort

In [21]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps
  
  def RECORDREADER(split):
    for value in split:
        yield (value, None)
      
  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])
    
def MAP(value:int, _):
  yield (value, None)
  
def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)
  
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.011149065951520765)),
   (None, np.float64(0.10423193390624308)),
   (None, np.float64(0.13824527153818011)),
   (None, np.float64(0.16637428533094312)),
   (None, np.float64(0.17201944493516197)),
   (None, np.float64(0.18936441347625066)),
   (None, np.float64(0.19478536309567396)),
   (None, np.float64(0.2078135218553332)),
   (None, np.float64(0.20852813605520348)),
   (None, np.float64(0.21294045699902509)),
   (None, np.float64(0.2234335256776936)),
   (None, np.float64(0.2391761985004981)),
   (None, np.float64(0.34906723405639895)),
   (None, np.float64(0.3544644011308551)),
   (None, np.float64(0.37244456575433027)),
   (None, np.float64(0.43764035061504136)),
   (None, np.float64(0.4479155857553083)),
   (None, np.float64(0.4544181622345699)),
   (None, np.float64(0.45671894226875065)),
   (None, np.float64(0.47219101481018566))]),
 (1,
  [(None, np.float64(0.5055204014172826)),
   (None, np.float64(0.5569647591397673)),
   (None, np.float64(0.6314

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [22]:
np.random.seed(42)
input_numbers = np.random.randint(low=0, high=1000, size=30).tolist()

def RECORDREADER():
    for (idx, num) in enumerate(input_numbers):
        yield (idx, num)   

def MAP(idx: int, value: int):
    yield ("max", value)  

def REDUCE(key: str, values: Iterator[int]):
    yield (key, max(values))  

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('max', 955)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [23]:
from typing import Tuple

def RECORDREADER():
    for (idx, num) in enumerate(input_numbers):
        yield (idx, num) 

def MAP(idx: int, value: float):
    yield ("mean", (value, 1))

def REDUCE(key: str, values: Iterator[Tuple]):
    total_sum, total_count = 0, 0
    for (s, c) in values:
        total_sum   += s   
        total_count += c   
    yield (key, total_sum / total_count)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('mean', 410.3333333333333)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [24]:
import itertools

def groupbykey_sort(iterable):
    pairs = sorted(iterable, key=lambda x: x[0])
    result = []
    for key, group in itertools.groupby(pairs, key=lambda x: x[0]):
        values = [v for (_, v) in group]
        result.append((key, values))
    return result

input = [('b',2), ('a',1), ('c',3), ('a',4), ('b',5), ('a',6)]
print(groupbykey_sort(input))

[('a', [1, 4, 6]), ('b', [2, 5]), ('c', [3])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [25]:
input_list = [3,1,4,1,5,9,2,6,5,3,5,8,9,7,9,3,2,3,8,4]

maps     = 4  
reducers = 3 

def INPUTFORMAT():
    def RECORDREADER(split):
        for (idx, element) in enumerate(split):
            yield (idx, element)  
    global maps
    split_size = int(np.ceil(len(input_list) / maps))
    for i in range(0, len(input_list), split_size):
        yield RECORDREADER(input_list[i : i + split_size])


def MAP(idx: int, element):
    yield (element, element)


def COMBINER(element, duplicates: Iterator):
    yield (element, element)


def PARTITIONER(key):
    global reducers
    return hash(key) % reducers


def REDUCE(element, duplicates: Iterator):
    yield (element, None)

result = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER, COMBINER)
output = sorted([k for (pid, part) in result for (k, _) in part])
print(output)

17 key-value pairs were sent over a network.
[1, 2, 3, 4, 5, 6, 7, 8, 9]


# Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [26]:
from typing import NamedTuple, Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

# Предикат C
def C(t: User) -> bool:
    return t.gender == "female" and t.social_contacts > 300

def RECORDREADER():
    for t in input_collection:
        yield (t.id, t)          

def MAP(_, t: NamedTuple):
    if C(t):                      
        yield (t, t)      
               
def REDUCE(t: NamedTuple, values: Iterator):
    yield (t, t)                 

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print([v for (_, v) in output])

[User(id=2, age=25, social_contacts=500, gender='female'), User(id=3, age=33, social_contacts=800, gender='female')]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [27]:
from typing import NamedTuple, Iterator

class User(NamedTuple):
    id:              int
    age:             int
    social_contacts: int
    gender:          str

input_collection = [
    User(id=0, age=55, gender='male',   social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800),
]

# Множество атрибутов S
S = {'age', 'gender'}

def project(t: NamedTuple, attrs: set) -> tuple:
    return tuple(
        getattr(t, field)
        for field in t._fields
        if field in attrs
    )

def RECORDREADER():
    for t in input_collection:
        yield (t.id, t)           

def MAP(_, t: NamedTuple):
    t_prime = project(t, S)      
    yield (t_prime, t_prime)     

def REDUCE(t_prime: tuple, values: Iterator):
    yield (t_prime, t_prime)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[((55, 'male'), (55, 'male')), ((25, 'female'), (25, 'female')), ((33, 'female'), (33, 'female'))]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [28]:
from typing import NamedTuple, Iterator

class User(NamedTuple):
    id:              int
    age:             int
    social_contacts: int
    gender:          str

R = [
    User(id=0, age=55, gender='male',   social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=33, gender='female', social_contacts=800),
]

S = [
    User(id=1, age=25, gender='female', social_contacts=240),  
    User(id=3, age=41, gender='male',   social_contacts=130),
    User(id=4, age=29, gender='female', social_contacts=310),
]

def RECORDREADER():
    for t in R:
        yield (t.id, t)
    for t in S:
        yield (t.id, t)

def MAP(_, t: NamedTuple):
    yield (t, t)             

def REDUCE(t: NamedTuple, values: Iterator):
    yield (t, t)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print([t for (t, t) in output])


[User(id=0, age=55, social_contacts=20, gender='male'), User(id=1, age=25, social_contacts=240, gender='female'), User(id=2, age=33, social_contacts=800, gender='female'), User(id=3, age=41, social_contacts=130, gender='male'), User(id=4, age=29, social_contacts=310, gender='female')]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [29]:
from typing import NamedTuple, Iterator

class User(NamedTuple):
    id:              int
    age:             int
    social_contacts: int
    gender:          str

R = [
    User(id=0, age=55, gender='male',   social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=33, gender='female', social_contacts=800),
]

S = [
    User(id=1, age=25, gender='female', social_contacts=240),  
    User(id=3, age=41, gender='male',   social_contacts=130),
    User(id=4, age=29, gender='female', social_contacts=310),
]

def RECORDREADER():
    for t in R:
        yield (t.id, t)
    for t in S:
        yield (t.id, t)

def MAP(_, t: NamedTuple):
    yield (t, t)                  

def REDUCE(t: NamedTuple, values: Iterator):
    values = list(values)
    if len(values) == 2:         
        yield (t, t)              
                                                           
output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print([t for (t, t) in output])

[User(id=1, age=25, social_contacts=240, gender='female')]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [30]:
from typing import NamedTuple, Iterator

class User(NamedTuple):
    id:              int
    age:             int
    social_contacts: int
    gender:          str

R = [
    User(id=0, age=55, gender='male',   social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=33, gender='female', social_contacts=800),
]

S = [
    User(id=1, age=25, gender='female', social_contacts=240),  
    User(id=3, age=41, gender='male',   social_contacts=130),
    User(id=4, age=29, gender='female', social_contacts=310),
]

def RECORDREADER():
    for t in R:
        yield (t.id, t)
    for t in S:
        yield (t.id, t)

def MAP(_, t: NamedTuple):
    if t in R:
        yield (t, 'R')       
    if t in S:
        yield (t, 'S')       

def REDUCE(t: NamedTuple, values: Iterator):
    values = list(values)
    if values == ['R']:       
        yield (t, t)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print([t for (t, _) in output])


[User(id=0, age=55, social_contacts=20, gender='male'), User(id=2, age=33, social_contacts=800, gender='female')]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [31]:
from typing import NamedTuple, Iterator, Tuple

class R_tuple(NamedTuple):   
    a: str                   
    b: str                   

class S_tuple(NamedTuple):  
    b: str                   
    c: str                  

R = [
    R_tuple(a="Alice", b="IT"),
    R_tuple(a="Bob",   b="HR"),
    R_tuple(a="Carol", b="IT"),
    R_tuple(a="Dave",  b="Finance"),
    R_tuple(a="Eve",   b="IT"),
]

S = [
    S_tuple(b="IT",      c="Moscow"),
    S_tuple(b="HR",      c="Berlin"),
    S_tuple(b="Finance", c="London"),
]

def RECORDREADER():
    for t in R:
        yield (t.a, t)
    for t in S:
        yield (t.b, t)

def MAP(_, t: NamedTuple):
    if isinstance(t, R_tuple):
        yield (t.b, ('R', t.a))    
    if isinstance(t, S_tuple):
        yield (t.b, ('S', t.c))  
        
def REDUCE(b: str, values: Iterator[Tuple]):
    values = list(values)
    from_R = [a for (rel, a) in values if rel == 'R']
    from_S = [c for (rel, c) in values if rel == 'S']
    for a in from_R:
        for c in from_S:
            yield (None, (a, b, c))

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print([v for (_, v) in output])


[('Alice', 'IT', 'Moscow'), ('Carol', 'IT', 'Moscow'), ('Eve', 'IT', 'Moscow'), ('Bob', 'HR', 'Berlin'), ('Dave', 'Finance', 'London')]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [32]:
from typing import NamedTuple, Iterator

class User(NamedTuple):
    id:              int    
    age:             int    
    social_contacts: int    
    gender:          str    

input_collection = [
    User(id=0, age=55, gender='male',   social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800),
    User(id=4, age=41, gender='male',   social_contacts=130),
    User(id=5, age=29, gender='male',   social_contacts=75),
]

def RECORDREADER():
    for t in input_collection:
        yield (t.id, t)

def MAP(_, t: User):
    yield (t.gender, t.social_contacts)

def REDUCE_SUM(a: str, values: Iterator[int]):
    yield (a, sum(values))

def REDUCE_MAX(a: str, values: Iterator[int]):
    yield (a, max(values))

def REDUCE_MIN(a: str, values: Iterator[int]):
    yield (a, min(values))

def REDUCE_COUNT(a: str, values: Iterator[int]):
    yield (a, sum(1 for _ in values))

def REDUCE_AVG(a: str, values: Iterator[int]):
    values = list(values)
    yield (a, sum(values) / len(values))

output = list(MapReduce(RECORDREADER, MAP, REDUCE_SUM))
print(output)


[('male', 225), ('female', 1540)]


# 

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [33]:
import numpy as np

np.random.seed(42)
I, J = 4, 5
M = np.random.randint(1, 10, (I, J))  # матрица на диске
v = np.random.randint(1, 10, J)       # вектор на диске 

# ПРОХОД 1 — join по j: перемножить M[i,j] и v[j]

def RECORDREADER_pass1():
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            yield ((i, j), ('M', i, int(M[i, j])))  # элемент матрицы
    for j in range(len(v)):
        yield (j, ('V', int(v[j])))                  # элемент вектора

def MAP_pass1(key, value: tuple):
    if value[0] == 'M':
        _, i, m_ij = value
        j = key[1]
        yield (j, ('M', i, m_ij))   
    if value[0] == 'V':
        j = key
        _, v_j = value
        yield (j, ('V', v_j))       

def REDUCE_pass1(j: int, values: Iterator):
    values = list(values)
    v_j = next(val for (tag, *val) in values if tag == 'V')[0]
    for item in values:
        if item[0] == 'M':
            _, i, m_ij = item
            yield (i, m_ij * v_j)   

pass1_output = list(MapReduce(RECORDREADER_pass1, MAP_pass1, REDUCE_pass1))

# ПРОХОД 2 — суммирование: x[i] = Σ_j M[i,j]·v[j]

def RECORDREADER_pass2():
    for (i, product) in pass1_output:
        yield (i, product)          # выход прохода 1 → вход прохода 2

def MAP_pass2(i: int, product: int):
    yield (i, product)              # передаём без изменений

def REDUCE_pass2(i: int, products: Iterator):
    yield (i, sum(products))        # суммируем слагаемые строки i

pass2_output = list(MapReduce(RECORDREADER_pass2, MAP_pass2, REDUCE_pass2))
result = [val for (_, val) in sorted(pass2_output)]
print(result)  

[129, 134, 104, 121]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$. 





In [34]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [35]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])
      
def MAP(k1, v1):
  (j, k) = k1
  w = v1
  # solution code that yield(k2,v2) 
  for i in range(small_mat.shape[0]):
    yield ((i, k), small_mat[i, j] * w)
  
def REDUCE(key, values):
  (i, k) = key
  # solution code that yield(k3,v3) pairs
  yield ((i, k), sum(values))

Проверьте своё решение

In [36]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat) 
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [37]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [40]:
np.random.seed(42)
I, J, K = 2, 3, 4

M = np.random.rand(I, J)  
N = np.random.rand(J, K)   
# ПРОХОД 1 - JOIN по j
def RECORDREADER_pass1():
    for i in range(M.shape[0]):         # сначала читаем M
        for j in range(M.shape[1]):
            yield ((i,j), ('M', i, j, float(M[i,j])))
    for j in range(N.shape[0]):         # затем читаем N
        for k in range(N.shape[1]):
            yield ((j,k), ('N', j, k, float(N[j,k])))

def MAP_pass1(k1, v1: tuple):
    tag = v1[0]
    if tag == 'M':
        _, i, j, v = v1
        yield (j, ('M', i, v))      # ключ = j, тег = 'M'
    if tag == 'N':
        _, j, k, w = v1
        yield (j, ('N', k, w))      # ключ = j, тег = 'N'

def REDUCE_pass1(j: int, values: Iterator):
    values = list(values)
    from_M = [(i, v) for (tag, i, v) in values if tag == 'M']
    from_N = [(k, w) for (tag, k, w) in values if tag == 'N']
    for (i, v) in from_M:
        for (k, w) in from_N:
            yield ((i, k), v * w)   

pass1_output = list(MapReduce(RECORDREADER_pass1, MAP_pass1, REDUCE_pass1))

# ПРОХОД 2 — суммирование: P[i,k] = Σ_j M[i,j]·N[j,k]

def RECORDREADER_pass2():
    for ((i, k), product) in pass1_output:
        yield ((i, k), product)     

def MAP_pass2(k1, v1):
    yield (k1, v1)               

def REDUCE_pass2(key: tuple, products: Iterator[float]):
    (i, k) = key
    yield ((i, k), sum(products))   # P[i,k] = Σ слагаемых

pass2_output = list(MapReduce(RECORDREADER_pass2, MAP_pass2, REDUCE_pass2))
pass2_output

[((0, 0), 0.17441939068744927),
 ((0, 1), 1.3807758790732332),
 ((0, 2), 1.2392602945033424),
 ((0, 3), 0.8511939466538538),
 ((1, 0), 0.06634751057614646),
 ((1, 1), 0.6984778135053759),
 ((1, 2), 0.5371992935787501),
 ((1, 3), 0.5388816433314578)]

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER. 

In [41]:
np.random.seed(42)
I, J, K  = 2, 3, 4
maps     = 3
reducers = 3

M = np.random.rand(I, J)
N = np.random.rand(J, K)

# Проход 1 - JOIN по j
# Каждая матрица в своём RECORDREADER

def INPUTFORMAT_pass1():
    def RECORDREADER_M():              
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                yield ((i,j), ('M', i, j, float(M[i,j])))

    def RECORDREADER_N():               
        for j in range(N.shape[0]):
            for k in range(N.shape[1]):
                yield ((j,k), ('N', j, k, float(N[j,k])))

    yield RECORDREADER_M()
    yield RECORDREADER_N()

def MAP_pass1(k1, v1):
    tag = v1[0]
    if tag == 'M':
        _, i, j, v = v1
        yield (j, ('M', i, v))        
    if tag == 'N':
        _, j, k, w = v1
        yield (j, ('N', k, w))         

def REDUCE_pass1(j, values):
    values = list(values)
    from_M = [(i, v) for (tag, i, v) in values if tag == 'M']
    from_N = [(k, w) for (tag, k, w) in values if tag == 'N']
    for (i, v) in from_M:
        for (k, w) in from_N:
            yield ((i,k), v * w)        

def PARTITIONER_pass1(key):
    return hash(key) % reducers

pass1_parts = list(MapReduceDistributed(
    INPUTFORMAT_pass1, MAP_pass1, REDUCE_pass1, PARTITIONER_pass1))
pass1_output = [(k,v) for (_,part) in pass1_parts for (k,v) in part]

# Проход 2 - суммирование по (i,k)

def INPUTFORMAT_pass2():
    chunk_size = int(np.ceil(len(pass1_output) / maps))
    for i in range(0, len(pass1_output), chunk_size):
        chunk = pass1_output[i : i + chunk_size]
        def RECORDREADER(chunk=chunk):
            for pair in chunk:
                yield pair
        yield RECORDREADER()

def MAP_pass2(k1, v1):
    yield (k1, v1)    
                     
def REDUCE_pass2(key, products):
    (i, k) = key
    yield ((i, k), sum(products))      

def PARTITIONER_pass2(key):
    return hash(key) % reducers

pass2_parts = list(MapReduceDistributed(
    INPUTFORMAT_pass2, MAP_pass2, REDUCE_pass2, PARTITIONER_pass2))

P = np.zeros((I, K))
for (_, part) in pass2_parts:
    for ((i,k), p_ik) in part:
        P[i, k] = p_ik
print(np.round(P, 4))


18 key-value pairs were sent over a network.
24 key-value pairs were sent over a network.
[[0.1744 1.3808 1.2393 0.8512]
 [0.0663 0.6985 0.5372 0.5389]]


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [43]:
np.random.seed(42)
I, J, K  = 3, 4, 5
maps_M   = 3
maps_N   = 2
reducers = 4
M = np.random.rand(I, J)
N = np.random.rand(J, K)

def split_indices(indices, n_workers):
    chunk_size = int(np.ceil(len(indices) / n_workers))
    return [indices[i:i+chunk_size] for i in range(0, len(indices), chunk_size)]

# СЛУЧАЙ 1: равномерное разбиение 
def INPUTFORMAT_pass1_uniform():
    # каждая матрица генерируется несколькими RECORDREADER-ами
    for chunk in split_indices(list(range(M.shape[0])), maps_M):
        def RECORDREADER_M(rows=chunk):
            for i in rows:
                for j in range(M.shape[1]):
                    yield ((i,j), ('M', i, j, float(M[i,j])))
        yield RECORDREADER_M()

    for chunk in split_indices(list(range(N.shape[1])), maps_N):
        def RECORDREADER_N(cols=chunk):
            for j in range(N.shape[0]):
                for k in cols:
                    yield ((j,k), ('N', j, k, float(N[j,k])))
        yield RECORDREADER_N()

# СЛУЧАЙ 2: случайное подмножество элементов 
def INPUTFORMAT_pass1_random():
    all_M = [(i,j) for i in range(M.shape[0]) for j in range(M.shape[1])]
    all_N = [(j,k) for j in range(N.shape[0]) for k in range(N.shape[1])]
    np.random.shuffle(all_M)   
    np.random.shuffle(all_N)

    for chunk in split_indices(all_M, maps_M):
        def RECORDREADER_M(indices=chunk):
            for (i,j) in indices:
                yield ((i,j), ('M', i, j, float(M[i,j])))
        yield RECORDREADER_M()

    for chunk in split_indices(all_N, maps_N):
        def RECORDREADER_N(indices=chunk):
            for (j,k) in indices:
                yield ((j,k), ('N', j, k, float(N[j,k])))
        yield RECORDREADER_N()

In [44]:
pass1_parts = list(MapReduceDistributed(
    INPUTFORMAT_pass1_uniform, MAP_pass1, REDUCE_pass1, PARTITIONER_pass1))
pass1_output = [(k,v) for (_,part) in pass1_parts for (k,v) in part]
pass2_parts = list(MapReduceDistributed(
    INPUTFORMAT_pass2, MAP_pass2, REDUCE_pass2, PARTITIONER_pass2))

P = np.zeros((I, K))
for (_, part) in pass2_parts:
    for ((i,k), p_ik) in part:
        P[i, k] = p_ik
print(np.round(P, 4))

32 key-value pairs were sent over a network.
60 key-value pairs were sent over a network.
[[1.3324 1.113  0.7066 1.5888 0.4948]
 [0.6741 0.6349 0.1405 0.6959 0.2285]
 [1.3767 1.0156 0.3699 1.1489 0.4512]]


In [45]:
pass1_parts = list(MapReduceDistributed(
    INPUTFORMAT_pass1_random, MAP_pass1, REDUCE_pass1, PARTITIONER_pass1))
pass1_output = [(k,v) for (_,part) in pass1_parts for (k,v) in part]
pass2_parts = list(MapReduceDistributed(
    INPUTFORMAT_pass2, MAP_pass2, REDUCE_pass2, PARTITIONER_pass2))

P = np.zeros((I, K))
for (_, part) in pass2_parts:
    for ((i,k), p_ik) in part:
        P[i, k] = p_ik
print(np.round(P, 4))

32 key-value pairs were sent over a network.
60 key-value pairs were sent over a network.
[[1.3324 1.113  0.7066 1.5888 0.4948]
 [0.6741 0.6349 0.1405 0.6959 0.2285]
 [1.3767 1.0156 0.3699 1.1489 0.4512]]
